# Free-Droid (Szabi) — v6 fine-tune (persona-bővítés + gazdagabb RAG + Q4-only)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

**Mi új a v5-höz képest — a lever most a PERSONA-adat és a RAG, nem a hiperparaméter:**
- **Persona-bővítés:** +50 dedupolt példa (`freedroid-ext.json`) — köszönés/köszöntés, red-team
  (jailbreak, wifi-csatlakozás elutasítása, system-prompt-védés), identitás, és **kifejtős**
  (koherencia) válaszok. Ez a v5 három gyenge dimenziójára megy: identitás, provokáció, koherencia.
- **tény→RAG szétválasztás:** a maradék tiszta Yotengrit-lookup (5 db) kikerült a fine-tune
  datasetből (a súlyokban hallucinál — v5: „Yotengri"); a tényt a RAG szállítja. **Dataset 731→726**
  (train 653 / val 73), tool-arány 22.0%.
- **Gazdagabb RAG:** a `yotengrit.md` korpusz **34→49 chunk** (a v5-ben hiányzó/hallucinált tények
  betöltve: „Mi az a Yotengrit?", jószomszédság, táltos, Boldogasszony, ünnepek, Mátün, …).
- **Q8 dobva:** a 3B **csak Q4_K_M**-et exportál (edge = Pi 5), a 8B is Q4 (cloud). A Q4/Q8 A/B lezárva.
- **Recept + nyelvtan változatlan:** `--preset gentle`, pozicionális `<tool>` nyelvtan.

**Először: Runtime → Change runtime type → T4 GPU.**

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## Repo klónozása

A v6 dataset (persona-bővítés + tény→RAG split) + a gazdagabb RAG-korpusz a **`main`-en** van
(a v6 PR mergelve). A notebook a main-t klónozza. (Merge ELŐTT: írd át a `-b main`-t a feature-branchre.)

In [ ]:
# 2. Repo a main-ről, majd be a training/-be. Guard: 726 példa, pozicionális nyelvtan, friss system_prompt.
!git clone --depth 1 -b main https://github.com/pits2022/free-droid.git
%cd free-droid/training
!python -c "import json, re; d=json.load(open('dataset/freedroid_full.json')); t=[x for x in d if '<tool>' in x['output']]; c=[x for x in t if x['output'].count('<tool>')>=2]; fn=[x for x in t if re.search(r'<tool>\s*[a-z_]+\(', x['output'])]; sp=open('system_prompt.txt', encoding='utf-8').read(); assert len(d)==726, f'VART 726 pelda (v6), de {len(d)} — rossz branch/merge?'; assert not fn, f'REGI fn() nyelvtan {len(fn)} peldaban — merge kell?'; assert 'move forward 2' in sp, 'system_prompt.txt nem pozicionalis — merge kell'; print(f'OK v6 | peldak: {len(d)} | tool: {len(t)} ({100*len(t)/len(d):.1f}%) | osszetett: {len(c)}')"
!wc -l dataset/train.jsonl dataset/val.jsonl

## Edge modell — Llama 3.2 3B (offline fallback)

A Pi 5-ön, teljesen offline futó modell. **Csak Q4_K_M** export (a Q8-at a config már nem exportálja).
Kimenet: `outputs/llama3.2-3b-v6/`.

In [ ]:
!python finetune.py --variant llama --preset gentle --tag v6

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A CAX31-en CPU-n futó nagyobb modell. A `llama8b` variáns T4-biztos (batch=1, grad_accum=8) és Q4_K_M-et
exportál. Kimenet: `outputs/llama3.1-8b-v6/`.

In [ ]:
!python finetune.py --variant llama8b --preset gentle --tag v6

## Next

- Kimenetek: `training/outputs/<variant>-v6/gguf-q4_k_m` (edge/cloud) + `lora-adapter`. Töltsd le (git-ignorált).
- **Ollama Modelfile (v6):** `SYSTEM` = a friss `training/system_prompt.txt` (pozicionális formátum!),
  `temperature 0.7`, `num_ctx 2048`, Llama-3 `TEMPLATE` + stop tokenek (a root `training/Modelfile` már
  Llama-helyes sablon; vagy minta: `tests/v5/**/Modelfile_*`). `ollama create szabi-3b-v6 -f Modelfile`
  / `szabi-8b-v6`. **Nincs külön Q8 oszlop** — a Q4/Q8 A/B lezárva.
- **Benchmark (`--json-out` KELL, hogy a judge tudja pontozni!):**
  `run_benchmark.py --models szabi-8b-v6 llama3.1:8b szabi-3b-v6 llama3.2:3b --rag --json-out`,
  majd `judge_benchmark.py`. Fő kérdés: a persona-bővítés behozta-e a **identitást / provokációt /
  koherenciát** (v5 gyenge pontjai), a tool_calling megtartásával; és a gazdagabb RAG csökkenti-e a
  „Yotengri" hallucinációt.
- **Kötelező red-team pass** a demó előtt. A tényeket futásidőben a `freedroid.rag` (49 chunk) adja.